<a href="https://colab.research.google.com/github/jeyner99/Integraci-n-de-datos-y-prospectiva/blob/main/Parcial_1_Integraci%C3%B3n_de_datos_y_prospectiva_Brainer_Arango%2C_Jeyner_Casta%C3%B1o.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
def caracterizacion(LDA):

    NI = 10 #Número de intervalos en los cuales quiero agrupar los datos
    counts, Limits = np.histogram(LDA, bins = NI) #Agrupar los datos en 10 intervalos
    LI = Limits[:-1] #Limites inferiores del histograma
    LS = Limits[1:]  #Limites superiores del histograma
    fi = counts/sum(counts) #Porcentaje de datos por intervalo
    MC = (LI+LS)/2 #Marca de clase: Dato representativo de cada intervalo

    df = pd.DataFrame(np.column_stack((LI, LS, counts, fi, MC)))
    df.columns = ["LI", "LS", "ND", "fi", "Marca de clase"]
    display(df)

    #Media - Valor esperado de una pérdida (Qué es lo más común que pase)
    u = np.sum(MC*fi)
    print("La media de las pérdidas es:", u)

    #Varianza - Indica que tan dispersos están los datos alrededor de la media
    varc = np.sum(((MC - u)**2)*fi)
    print("La varianza de las pérdidas es:", varc)

    #Desviación estándar
    sigma = np.sqrt(varc)
    print("La desviación estándar de las pérdidas es:", sigma)

    #Intervalo de variación de las pérdidas
    #En este intervalo se encuentra el 95% de los datos (2*sigma)
    print("El limite inferior es ", u-2*sigma)
    print("El limite superior es ", u+2*sigma)

    #Coeficiente de asimetría
    coef_asm = np.sum((((MC-u)**3)*fi)/(sigma**3))
    print("El coeficiente de asimetría es:", coef_asm)

    #Coeficiente de curtosis
    curtosis = np.sum((((MC-u)**4*fi))/(sigma**4))-3
    coef_curt = np.sum((((MC-u)**4)*fi)/((sigma**4)))
    print("El coeficiente de curtosis es:", curtosis)

#Caso de estudio LATAM AIRLINES

LATAM Airlines Group es el mayor grupo aéreo de América Latina, con presencia en múltiples países de la región y una flota que, al cierre de 2025, alcanzó 371 aeronaves, proyectando llegar a 410 durante 2026, lo que la posiciona entre las 12 aerolíneas con mayor flota del mundo.

Como práctica comercial habitual en la industria, LATAM vende más tiquetes que asientos disponibles (overbooking), asumiendo el riesgo de que se presenten más pasajeros que cupos, con el fin de compensar el efecto de los “no-shows” (pasajeros que no se presentan al vuelo) y maximizar la ocupación de sus aeronaves.

La empresa está interesada en desarrollar un sistema por adaptación y aprendizaje que le permita proyectar, desde la gestión de sus riesgos, el porcentaje óptimo de sobreventa por vuelo y por ruta, minimizando tanto los asientos vacíos como las compensaciones por denegación de embarque.

Las variables que serán utilizadas para este proceso son las siguientes:

Vuelos semanales: Indica la cantidad de vuelos que realizó la aerolínea en esa semana.

Overbooking: Es la cantidad de veces que se materializó el riesgo de overbooking, es decir las veces que llegaron más personas de los cupos que habían para el vuelo.

Compensación: Es el dinero que gastó la aerolínea en esa semana compensando a los pasajeros que se quedaron sin vuelo.

Crearemos nuestra base de datos dentro de nuestro código con valores semialeatorios.

In [ ]:
# Añadimos una random seed para que los resultados sigan siendo los mismos de la tabla enviada cuando se vuelva a correr el código
np.random.seed(42)

# Number of observations
n_observations = 700

# Generate 'Semana' column
new_semana = np.arange(1, n_observations + 1)

# Generate 'Transacción Semanales' (number of flights per week)
# Similar range to original, but for flights
new_transacciones_semanales = np.random.randint(50, 500, n_observations)

# Generate 'Frecuencia' (number of overbooking incidents)
# Keeping incidents relatively low compared to flights
new_frecuencia = np.random.randint(1, 50, n_observations)

# Generate 'Severidad' (cost per incident in USD) with positive skewness
# Using a log-normal distribution for cost tends to give positive skewness
# Mean and sigma for the underlying normal distribution adjusted for reasonable USD values
new_severidad = np.random.lognormal(mean=6.5, sigma=0.8, size=n_observations) # Values will be in USD

# Create the new DataFrame
df_aerolinea_LATAM = pd.DataFrame({
    'Semana': new_semana,
    'Vuelos semanales': new_transacciones_semanales,
    'Overbooking': new_frecuencia,
    'Compensación': new_severidad
})

# Display the first 5 rows of the new DataFrame
display(df_aerolinea_LATAM.head())

# Save the DataFrame to an Excel file
df_aerolinea_LATAM.to_excel('df_aerolinea_LATAM.xlsx', index=False)
print("DataFrame 'df_aerolinea' guardado como 'df_aerolinea_LATAM.xlsx'")

In [ ]:
Freq = df_aerolinea_LATAM.iloc[:,2]
Sev = df_aerolinea_LATAM.iloc[:,3]
LDA = Freq*Sev
ND=len(LDA)
LDA=Freq*Sev
sigmao=np.std(LDA)
uo=np.mean(LDA)


plt.figure()
sns.histplot(LDA, color="blue", bins=10, kde = True)
plt.title("Distribución agregada de las pérdidas - LATAM Airlines")
plt.grid()
plt.show()


In [ ]:
caracterizacion(LDA)

#Mejorar la confiabilidad mediante métodos de integración.

El primer método será muestreo aleatorio con el método de montecarlo.

In [ ]:
import random #Cogemos cinco datos random de los que tenemos
random.seed(42)
XC=np.array(random.choices(LDA, k=5))
XC=np.sort(XC)
print("Los concentradores de información son:", XC)

In [ ]:
Box0=[]; Box1=[];Box2=[];Box3=[];Box4=[]
Boxes=[Box0,Box1,Box2,Box3,Box4]
for k in range(len(LDA)):
  d=np.abs(XC-LDA.iloc[k,])/XC #La distancia de un dato a un concentrador
  nc=np.argmin(d)#Numero de la caja donde irá el dato
  #print("La distancia porcentual es de:", d)
  #print("El dato pertenece a la categoria:", nc)
  Boxes[nc].append(LDA.iloc[k,])

# Determinamos el número de datos a muestrear por caja una vez que todas las Boxes están llenas
pm = [] # Vector para almacenar el número de muestras por caja
for j in range(5):
    # Calculamos la proporción de datos en la caja actual
    proportion = len(Boxes[j]) / len(LDA)
    # Escalar esta proporción por la longitud total de LDA para obtener el número de muestras.
    # Usamos round para manejar posibles problemas de punto flotante y lo convertimos a entero.
    num_samples = int(round(proportion * len(LDA)))
    pm.append(num_samples)
pm = np.array(pm) # Convertir a un array de numpy
print("El número de datos a muestrear de cada caja es:", pm)

In [ ]:
LDA2=[]

for j in range(5):
    if len(Boxes[j]) > 0:
        LDA2.extend(random.choices(Boxes[j], k=pm[j]))

LDA2=np.array(LDA2)

#Se comprueban las metricas estadisticas
uo=np.mean(LDA)
print("La media de los datos observados es:", uo)
ue=np.mean(LDA2)
print("La media de los datos externos es:", ue)
du=np.abs((uo-ue)/uo)
print("El error relativo porcentual es:", du*100) # Corregido: multiplicado por 100 para porcentaje

plt.figure()
sns.kdeplot(LDA, color="red", label="Datos observados")
sns.kdeplot(LDA2, color="blue", label="Datos externos")
plt.grid()
plt.show()

In [ ]:
import numpy as np
import pandas as pd

# Asegurarse de que LDA sea un array de numpy o una lista para facilitar la concatenación
# Si LDA es una Serie de Pandas, se convierte a array de numpy
if isinstance(LDA, pd.Series):
    LDA_array = LDA.values
else:
    LDA_array = np.asarray(LDA)

# Seleccionar 300 datos aleatorios de LDA2
# np.random.choice es útil para muestreo aleatorio.
# replace=True permite que el mismo elemento sea seleccionado más de una vez si es necesario,
# pero dado que estamos integrando, y no solo reemplazando, es adecuado.
# Aquí asumimos que no es un problema si los 300 datos se repiten si LDA2 es pequeño,
# pero LDA2 tiene 700 elementos, así que 300 es seguro sin repetición.
# Para evitar repeticiones si la muestra es menor que la población, se usa replace=False
# Sin embargo, random.choice con replace=True es más general si alguna vez se pidiera más elementos que la longitud de LDA2
sampled_data = np.random.choice(LDA2, size=300, replace=False)

# Integrar los datos muestreados a LDA
LDA = np.concatenate((LDA_array, sampled_data))

# Opcionalmente, puedes volver a convertir LDA a una Serie de Pandas si es como se usa habitualmente.
LDA = pd.Series(LDA)

print(f"Se han integrado 300 datos de LDA2 a LDA. El nuevo tamaño de LDA es: {len(LDA)}")

In [ ]:
caracterizacion(LDA)

Lo que se hizo fue incluir 300 datos nuevos a LDA, esto para mejorar la confiabilidad de los datos al tener 1000 en vez de 700, podemos observar cuando imprimimos las nuevas medidas de tendencia de LDA que la variación después de incluir los nuevos datos es muy baja, lo cual es una buena señal porque quiere decir que mejoramos la confiabilidad sin perder la esencia original de nuestros datos, siguen siendo muy afines a la realidad.

In [ ]:
np.random.seed(84) # Seed diferente para el dataset externo
n_observations_e = 800

#Generamos semana ademas de vuelos y overcooking con rangos distintos
semana_e = np.arange(1, n_observations_e + 1)
transacciones_semanales_e = np.random.randint(30, 400, n_observations_e)
frecuencia_e = np.random.randint(0, 40, n_observations_e)

severidad_e = np.random.lognormal(mean=6.0, sigma=0.7, size=n_observations_e) # Distribución de costos ligeramente diferente

# Creamos el df
df_aerolinea_Avianca = pd.DataFrame({
    'Semana': semana_e,
    'Vuelos semanales': transacciones_semanales_e,
    'Overbooking': frecuencia_e,
    'Compensación': severidad_e
})

#Mostramos el df y lo guardamos como excel
display(df_aerolinea_Avianca.head())
XDe=df_aerolinea_Avianca

# Save the DataFrame to an Excel file
df_aerolinea_Avianca.to_excel('df_aerolinea_Avianca.xlsx', index=False)
print("DataFrame 'df_aerolinea' guardado como 'df_aerolinea_LATAM.xlsx'")

In [ ]:
#Ahora con el dataset externo
Freqe=XDe.iloc[:,2]; Seve=XDe.iloc[:,3] # Corregidos los índices de las columnas ya que no se cuenta con exactamente las mismas columnas (LI,LS)
LDAe=Freqe*Seve
NDe=len(LDAe)
sigmae=np.std(LDAe)
ue=np.mean(LDAe)
XDe.head()

In [ ]:
caracterizacion(LDAe)

In [ ]:
#Varianza ponderada
EPV=(ND*sigmao**2+NDe*sigmae**2)/(ND+NDe)
print("La ponderación de la varianza es:", EPV)

#Media hipotetica - ponderado de las medias
uh=(ND*uo+NDe*ue)/(ND+NDe)
print("La media hipotetica ponderada es:", uh)

#La varianza de dos variables es:
VHM=(((ND)/(ND+NDe))*uo**2+((NDe)/(ND+NDe))*ue**2)-uh**2
print("La varianza de dos variables es:", VHM)
Cr=ND/(ND+(EPV/VHM))
print("La credibilidad de mis datos es:",Cr*100)
print("La credibilidad de la base de datos externos es del:",(1-Cr)*100)